<a href="https://colab.research.google.com/github/WajihC/ccny-calendar-scraper/blob/main/ccny_calendar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Scraping the CCNY Fall 2021 Academic Calendar


import re
from datetime import date, datetime, timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd





In [2]:
url = "https://www.ccny.cuny.edu/registrar/fall"
headers = {"User-Agent": "Mozilla/5.0 (CCNY 46000 HW3 scraper)"}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()
print(response.status_code, len(response.text), "characters")

NameError: name 'URL' is not defined

In [3]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True))

NameError: name 'response' is not defined

In [ ]:
table = soup.find("table")
rows = table.find_all("tr")
print(len(rows), "rows in the table")



In [ ]:
def clean(cell):
    """Get a cell's text with non-breaking spaces and extra whitespace collapsed."""
    text = cell.get_text(" ", strip=True).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()

raw = [[clean(td) for td in tr.find_all(["td", "th"])] for tr in rows]
for r in raw[:6]:
    print(r)

In [ ]:
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_RE = re.compile(
    rf"^(?P<m1>{MONTHS})\s+(?P<d1>\d{{1,2}})(?:\s*,\s*(?P<y1>\d{{4}}))?"
    rf"(?:\s*[-–]\s*(?:(?P<m2>{MONTHS})\s+)?(?P<d2>\d{{1,2}})"
    rf"(?:\s*,\s*(?P<y2>\d{{4}}))?)?$",
    re.IGNORECASE,
)

TERM_YEAR = 2021

def to_date(month, day, year=None):
    if year is None:
        # Fall 2021 term: anything listed in January belongs to the next year
        year = TERM_YEAR + 1 if month.lower() == "january" else TERM_YEAR
    return datetime.strptime(f"{month} {day} {year}", "%B %d %Y").date()

def parse_dates(s):
    m = DATE_RE.match(s.strip())
    if not m:
        return []
    start = to_date(m["m1"], m["d1"], m["y1"] or m["y2"])
    if m["d2"] is None:
        return [start]
    end = to_date(m["m2"] or m["m1"], m["d2"], m["y2"] or m["y1"])
    return [start + timedelta(days=i) for i in range((end - start).days + 1)]

for r in raw[1:]:
    print(r[0], "->", parse_dates(r[0]))

In [ ]:
assert parse_dates("August 01") == [date(2021, 8, 1)]
assert parse_dates("August 25 - 31")[-1] == date(2021, 8, 31)
assert len(parse_dates("August 25 - 31")) == 7
assert parse_dates("January 1, 2022") == [date(2022, 1, 1)]
assert parse_dates("DATES") == []          # header row is skipped
print("date parser OK")

In [ ]:
def clean_text(s):
    parts = [p.strip() for p in s.split(";")]
    return "; ".join(p for p in parts if p)

print(clean_text("Start of Fall Term;  Classes begin;  Initial Registration Appeals begin;"))

In [ ]:
records = []
mismatches = []                         # sanity check against the site's DAYS column

for tr in rows:
    cells = [clean(td) for td in tr.find_all(["td", "th"])]
    if len(cells) < 3:
        continue
    date_str, days_str, text = cells[0], cells[1], cells[2]
    dates = parse_dates(date_str)
    if not dates:
        continue

    # The site says e.g. 'Wednesday - Tuesday' for a range: check first and last day
    site_names = [n.strip() for n in re.split(r"[-–]", days_str)]
    if (dates[0].strftime("%A"), dates[-1].strftime("%A")) != (site_names[0], site_names[-1]):
        mismatches.append((date_str, days_str))

    for d in dates:
        records.append({"date": d, "dow": d.strftime("%A"), "text": clean_text(text)})

print(len(records), "records")
print("weekday mismatches with the site:", mismatches or "none")